In [18]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, 
                             f1_score, accuracy_score, classification_report)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG19, InceptionV3
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

# Enable eager execution
tf.config.run_functions_eagerly(True)

In [19]:
TRAIN_DIR = "Jute_Pest_Dataset_Split/train"
VALID_DIR = "Jute_Pest_Dataset_Split/val"
TEST_DIR = "Jute_Pest_Dataset_Split/test"

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 0.001

In [20]:
# Data generators with augmentation (following the research paper)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

# Load datasets
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    VALID_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

num_classes = len(train_generator.class_indices)
print(f"Number of classes: {num_classes}")
print(f"Class indices: {train_generator.class_indices}")


Found 5005 images belonging to 17 classes.
Found 1073 images belonging to 17 classes.
Found 1102 images belonging to 17 classes.
Number of classes: 17
Class indices: {'Beet Armyworm': 0, 'Black Hairy': 1, 'Cutworm': 2, 'Field Cricket': 3, 'Jute Aphid': 4, 'Jute Hairy': 5, 'Jute Red Mite': 6, 'Jute Semilooper': 7, 'Jute Stem Girdler': 8, 'Jute Stem Weevil': 9, 'Leaf Beetle': 10, 'Mealybug': 11, 'Pod Borer': 12, 'Scopula Emissaria': 13, 'Termite': 14, 'Termite odontotermes (Rambur)': 15, 'Yellow Mite': 16}


In [21]:
def build_pretrained_model(base_model, num_classes, model_name):
    """
    Build fine-tuned pre-trained model following the research paper
    - Replace classification layer with global average pooling
    - Add dropout layer (30%)
    - Add dense layer with softmax
    """
    # Freeze base model layers
    base_model.trainable = False
    
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

# Load pre-trained models
print("\n" + "="*70)
print("Loading Pre-trained Models...")
print("="*70)

# VGG19 Model
vgg19_base = VGG19(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
vgg19_model = build_pretrained_model(vgg19_base, num_classes, 'VGG19')

# InceptionV3 Model
inception_base = InceptionV3(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
inception_model = build_pretrained_model(inception_base, num_classes, 'InceptionV3')


Loading Pre-trained Models...


In [22]:
optimizer = Adam(learning_rate=LEARNING_RATE)
loss_fn = keras.losses.CategoricalCrossentropy()

# VGG19
vgg19_model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# InceptionV3
inception_model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

print("\n" + "="*70)
print("Model Summaries")
print("="*70)
print("\nVGG19:")
vgg19_model.summary()
print("\n\nInceptionV3:")
inception_model.summary()


Model Summaries

VGG19:
Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vgg19 (Functional)          (None, 7, 7, 512)         20024384  
                                                                 
 global_average_pooling2d_4  (None, 512)               0         
  (GlobalAveragePooling2D)                                       
                                                                 
 dropout_4 (Dropout)         (None, 512)               0         
                                                                 
 dense_4 (Dense)             (None, 17)                8721      
                                                                 
Total params: 20033105 (76.42 MB)
Trainable params: 8721 (34.07 KB)
Non-trainable params: 20024384 (76.39 MB)
_________________________________________________________________


InceptionV3:
Model: "sequential_5"
______________

In [26]:

print("\n" + "="*70)
print("Setting up Callbacks...")
print("="*70)

# Create directory for checkpoints
os.makedirs('checkpoints', exist_ok=True)

# Callbacks for VGG19
vgg19_early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=15,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

vgg19_reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.5,
    patience=10,
    mode='max',
    min_lr=1e-7,
    verbose=1
)

vgg19_checkpoint = keras.callbacks.ModelCheckpoint(
    'checkpoints/vgg19_best_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

vgg19_callbacks = [vgg19_early_stopping, vgg19_reduce_lr, vgg19_checkpoint]

# Callbacks for InceptionV3
inception_early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=15,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

inception_reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.5,
    patience=10,
    mode='max',
    min_lr=1e-7,
    verbose=1
)

inception_checkpoint = keras.callbacks.ModelCheckpoint(
    'checkpoints/inception_v3_best_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

inception_callbacks = [inception_early_stopping, inception_reduce_lr, inception_checkpoint]

print("✓ Callbacks configured successfully")


Setting up Callbacks...
✓ Callbacks configured successfully


In [27]:
print("\n" + "="*70)
print("Training Pre-trained Models...")
print("="*70)

# Train InceptionV3
print("\nTraining InceptionV3...")
inception_history = inception_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    verbose=1
)






Training Pre-trained Models...

Training InceptionV3...
Epoch 1/20
313/313 [==============================] - 625s 2s/step - loss: 0.4157 - accuracy: 0.8711 - val_loss: 0.2454 - val_accuracy: 0.9273
Epoch 2/20
313/313 [==============================] - 600s 2s/step - loss: 0.4203 - accuracy: 0.8745 - val_loss: 0.2962 - val_accuracy: 0.9068
Epoch 3/20
313/313 [==============================] - 618s 2s/step - loss: 0.4342 - accuracy: 0.8735 - val_loss: 0.2140 - val_accuracy: 0.9236
Epoch 4/20
313/313 [==============================] - 596s 2s/step - loss: 0.4105 - accuracy: 0.8761 - val_loss: 0.2256 - val_accuracy: 0.9226
Epoch 5/20
313/313 [==============================] - 600s 2s/step - loss: 0.3891 - accuracy: 0.8795 - val_loss: 0.2383 - val_accuracy: 0.9292
Epoch 6/20
313/313 [==============================] - 574s 2s/step - loss: 0.3742 - accuracy: 0.8837 - val_loss: 0.2297 - val_accuracy: 0.9348
Epoch 7/20
313/313 [==============================] - 554s 2s/step - loss: 0.3957 - a

KeyboardInterrupt: 

In [ ]:
# Re-initialize the optimizer for VGG19
optimizer = Adam(learning_rate=LEARNING_RATE)
vgg19_model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

# Train VGG19
print("\nTraining VGG19...")

# Enable eager execution
tf.config.run_functions_eagerly(True)

vgg19_history = vgg19_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    verbose=1
)

In [ ]:


def build_pretrained_model(base_model, num_classes, model_name):
    """
    Build fine-tuned pre-trained model following the research paper
    - Replace classification layer with global average pooling
    - Add dropout layer (30%)
    - Add dense layer with softmax
    """
    # Freeze base model layers
    base_model.trainable = False
    
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

# Load pre-trained models
print("\n" + "="*70)
print("Loading Pre-trained Models...")
print("="*70)

# VGG19 Model
vgg19_base = VGG19(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
vgg19_model = build_pretrained_model(vgg19_base, num_classes, 'VGG19')

# InceptionV3 Model
inception_base = InceptionV3(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
inception_model = build_pretrained_model(inception_base, num_classes, 'InceptionV3')

# ============================================================================
# 3. COMPILE MODELS
# ============================================================================

optimizer = Adam(learning_rate=LEARNING_RATE)
loss_fn = keras.losses.CategoricalCrossentropy()

# VGG19
if vgg19_model is not None:
    vgg19_model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])
    print("✓ VGG19 compiled successfully")

# InceptionV3
if inception_model is not None:
    inception_model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])
    print("✓ InceptionV3 compiled successfully")

print("\n" + "="*70)
print("Model Summaries")
print("="*70)

if vgg19_model is not None:
    print("\nVGG19:")
    vgg19_model.summary()

if inception_model is not None:
    print("\n\nInceptionV3:")
    inception_model.summary()

# ============================================================================
# 3.5 SETUP CALLBACKS
# ============================================================================

print("\n" + "="*70)
print("Setting up Callbacks...")
print("="*70)

# Create directory for checkpoints
os.makedirs('checkpoints', exist_ok=True)

# Callbacks for VGG19
vgg19_early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=15,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

vgg19_reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.5,
    patience=10,
    mode='max',
    min_lr=1e-7,
    verbose=1
)

vgg19_checkpoint = keras.callbacks.ModelCheckpoint(
    'checkpoints/vgg19_best_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

vgg19_callbacks = [vgg19_early_stopping, vgg19_reduce_lr, vgg19_checkpoint]

# Callbacks for InceptionV3
inception_early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=15,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

inception_reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.5,
    patience=10,
    mode='max',
    min_lr=1e-7,
    verbose=1
)

inception_checkpoint = keras.callbacks.ModelCheckpoint(
    'checkpoints/inception_v3_best_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

inception_callbacks = [inception_early_stopping, inception_reduce_lr, inception_checkpoint]

print("✓ Callbacks configured successfully")

# ============================================================================
# 4. TRAIN MODELS
# ============================================================================

print("\n" + "="*70)
print("Training Pre-trained Models...")
print("="*70)

# Train VGG19
if vgg19_model is not None:
    print("\nTraining VGG19...")
    vgg19_history = vgg19_model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=EPOCHS,
        verbose=1
    )
else:
    vgg19_history = None

# Train InceptionV3
if inception_model is not None:
    print("\nTraining InceptionV3...")
    try:
        inception_history = inception_model.fit(
            train_generator,
            validation_data=val_generator,
            epochs=EPOCHS,
            verbose=1,
            use_multiprocessing=False
        )
    except NotImplementedError as e:
        print(f"Error training InceptionV3: {e}")
        print("Attempting workaround...")
        inception_history = inception_model.fit(
            train_generator,
            validation_data=val_generator,
            epochs=EPOCHS,
            verbose=1
        )
else:
    inception_history = None

# ============================================================================
# 6. PLOT TRAINING HISTORY
# ============================================================================

def plot_training_history(history, model_name):
    """Plot training and validation loss/accuracy"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Loss
    axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title(f'{model_name} - Training and Validation Loss', fontsize=13, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
    axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy', fontsize=12)
    axes[1].set_title(f'{model_name} - Training and Validation Accuracy', fontsize=13, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{model_name}_training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

print("\n" + "="*70)
print("Plotting Training History...")
print("="*70)

if vgg19_history is not None:
    plot_training_history(vgg19_history, 'VGG19')

if inception_history is not None:
    plot_training_history(inception_history, 'InceptionV3')

# ============================================================================
# 7. EVALUATE ON TEST SET
# ============================================================================

print("\n" + "="*70)
print("Evaluating on Test Set...")
print("="*70)

# Get predictions
def get_predictions(model, test_gen):
    """Generate predictions for test dataset"""
    predictions = model.predict(test_gen)
    pred_classes = np.argmax(predictions, axis=1)
    true_classes = test_gen.classes
    return pred_classes, true_classes, predictions

vgg19_pred, vgg19_true, vgg19_probs = None, None, None
inception_pred, inception_true, inception_probs = None, None, None

if vgg19_model is not None:
    vgg19_pred, vgg19_true, vgg19_probs = get_predictions(vgg19_model, test_generator)

if inception_model is not None:
    inception_pred, inception_true, inception_probs = get_predictions(inception_model, test_generator)

# ============================================================================
# 8. COMPUTE EVALUATION METRICS
# ============================================================================

def compute_metrics(y_true, y_pred, model_name):
    """Compute and return evaluation metrics"""
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    metrics = {
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }
    
    return metrics

# Compute metrics for pre-trained models
vgg19_metrics = None
inception_metrics = None

if vgg19_pred is not None:
    vgg19_metrics = compute_metrics(vgg19_true, vgg19_pred, 'VGG19')

if inception_pred is not None:
    inception_metrics = compute_metrics(inception_true, inception_pred, 'InceptionV3')

# Display metrics comparison
print("\n" + "="*70)
print("PRE-TRAINED MODELS PERFORMANCE METRICS")
print("="*70)

metrics_list = []
if vgg19_metrics is not None:
    metrics_list.append([vgg19_metrics['Model'], f"{vgg19_metrics['Accuracy']:.4f}", 
     f"{vgg19_metrics['Precision']:.4f}", f"{vgg19_metrics['Recall']:.4f}", 
     f"{vgg19_metrics['F1-Score']:.4f}"])

if inception_metrics is not None:
    metrics_list.append([inception_metrics['Model'], f"{inception_metrics['Accuracy']:.4f}", 
     f"{inception_metrics['Precision']:.4f}", f"{inception_metrics['Recall']:.4f}", 
     f"{inception_metrics['F1-Score']:.4f}"])

metrics_df = np.array(metrics_list)

print(f"{'Model':<20} {'Accuracy':<15} {'Precision':<15} {'Recall':<15} {'F1-Score':<15}")
print("-" * 80)
for row in metrics_df:
    print(f"{row[0]:<20} {row[1]:<15} {row[2]:<15} {row[3]:<15} {row[4]:<15}")

# ============================================================================
# 9. CONFUSION MATRICES
# ============================================================================

def plot_confusion_matrix(y_true, y_pred, model_name, class_names):
    """Plot confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, 
                yticklabels=class_names, cbar_kws={'label': 'Count'})
    plt.title(f'{model_name} - Confusion Matrix', fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f'{model_name}_confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

class_names = list(train_generator.class_indices.keys())

print("\n" + "="*70)
print("Generating Confusion Matrices...")
print("="*70)

if vgg19_pred is not None:
    plot_confusion_matrix(vgg19_true, vgg19_pred, 'VGG19', class_names)

if inception_pred is not None:
    plot_confusion_matrix(inception_true, inception_pred, 'InceptionV3', class_names)

# ============================================================================
# 10. CLASSIFICATION REPORTS
# ============================================================================

print("\n" + "="*70)
print("CLASSIFICATION REPORTS")
print("="*70)

if vgg19_pred is not None:
    print("\n\nVGG19 Classification Report:")
    print(classification_report(vgg19_true, vgg19_pred, target_names=class_names, zero_division=0))

if inception_pred is not None:
    print("\n\nInceptionV3 Classification Report:")
    print(classification_report(inception_true, inception_pred, target_names=class_names, zero_division=0))

# ============================================================================
# 11. VISUALIZE METRICS COMPARISON
# ============================================================================

def plot_metrics_comparison():
    """Plot comparison of metrics across pre-trained models"""
    if vgg19_metrics is None and inception_metrics is None:
        print("No metrics available to plot")
        return
    
    models = []
    accuracy = []
    precision = []
    recall = []
    f1 = []
    
    if vgg19_metrics is not None:
        models.append('VGG19')
        accuracy.append(vgg19_metrics['Accuracy'])
        precision.append(vgg19_metrics['Precision'])
        recall.append(vgg19_metrics['Recall'])
        f1.append(vgg19_metrics['F1-Score'])
    
    if inception_metrics is not None:
        models.append('InceptionV3')
        accuracy.append(inception_metrics['Accuracy'])
        precision.append(inception_metrics['Precision'])
        recall.append(inception_metrics['Recall'])
        f1.append(inception_metrics['F1-Score'])
    
    x = np.arange(len(models))
    width = 0.2
    
    plt.figure(figsize=(10, 6))
    plt.bar(x - 1.5*width, accuracy, width, label='Accuracy', alpha=0.8)
    plt.bar(x - 0.5*width, precision, width, label='Precision', alpha=0.8)
    plt.bar(x + 0.5*width, recall, width, label='Recall', alpha=0.8)
    plt.bar(x + 1.5*width, f1, width, label='F1-Score', alpha=0.8)
    
    plt.xlabel('Model', fontsize=12, fontweight='bold')
    plt.ylabel('Score', fontsize=12, fontweight='bold')
    plt.title('Pre-trained Models Performance Comparison', fontsize=14, fontweight='bold')
    plt.xticks(x, models)
    plt.legend()
    plt.ylim([0, 1.1])
    plt.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i in range(len(models)):
        plt.text(i - 1.5*width, accuracy[i] + 0.02, f'{accuracy[i]:.3f}', ha='center', va='bottom', fontsize=9)
        plt.text(i - 0.5*width, precision[i] + 0.02, f'{precision[i]:.3f}', ha='center', va='bottom', fontsize=9)
        plt.text(i + 0.5*width, recall[i] + 0.02, f'{recall[i]:.3f}', ha='center', va='bottom', fontsize=9)
        plt.text(i + 1.5*width, f1[i] + 0.02, f'{f1[i]:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('metrics_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_metrics_comparison()

# ============================================================================
# 12. SAVE MODELS
# ============================================================================

print("\n" + "="*70)
print("Saving Models...")
print("="*70)

# Create directory for models if it doesn't exist
os.makedirs('saved_models', exist_ok=True)

# Save VGG19 model
if vgg19_model is not None:
    try:
        vgg19_model.save('saved_models/vgg19_model')
        print("✓ VGG19 model saved as 'saved_models/vgg19_model'")
    except Exception as e:
        print(f"✗ Error saving VGG19 model: {e}")

# Save InceptionV3 model
if inception_model is not None:
    try:
        inception_model.save('saved_models/inception_v3_model')
        print("✓ InceptionV3 model saved as 'saved_models/inception_v3_model'")
    except Exception as e:
        print(f"✗ Error saving InceptionV3 model: {e}")

# ============================================================================
# 13. SAVE TRAINING HISTORY
# ============================================================================

print("\n" + "="*70)
print("Saving Training History...")
print("="*70)

import pickle

# Save VGG19 history
if vgg19_history is not None:
    try:
        with open('saved_models/vgg19_history.pkl', 'wb') as f:
            pickle.dump(vgg19_history.history, f)
        print("✓ VGG19 training history saved as 'saved_models/vgg19_history.pkl'")
    except Exception as e:
        print(f"✗ Error saving VGG19 history: {e}")

# Save InceptionV3 history
if inception_history is not None:
    try:
        with open('saved_models/inception_v3_history.pkl', 'wb') as f:
            pickle.dump(inception_history.history, f)
        print("✓ InceptionV3 training history saved as 'saved_models/inception_v3_history.pkl'")
    except Exception as e:
        print(f"✗ Error saving InceptionV3 history: {e}")

# ============================================================================
# 14. SAVE EVALUATION METRICS
# ============================================================================

print("\n" + "="*70)
print("Saving Evaluation Metrics...")
print("="*70)

import json

metrics_summary = {}

if vgg19_metrics is not None:
    metrics_summary['VGG19'] = vgg19_metrics

if inception_metrics is not None:
    metrics_summary['InceptionV3'] = inception_metrics

try:
    with open('saved_models/evaluation_metrics.json', 'w') as f:
        json.dump(metrics_summary, f, indent=4)
    print("✓ Evaluation metrics saved as 'saved_models/evaluation_metrics.json'")
except Exception as e:
    print(f"✗ Error saving evaluation metrics: {e}")

# ============================================================================
# 15. SAVE CLASS INDICES
# ============================================================================

print("\n" + "="*70)
print("Saving Class Indices...")
print("="*70)

try:
    with open('saved_models/class_indices.json', 'w') as f:
        json.dump(train_generator.class_indices, f, indent=4)
    print("✓ Class indices saved as 'saved_models/class_indices.json'")
except Exception as e:
    print(f"✗ Error saving class indices: {e}")

print("\n" + "="*70)
print("ANALYSIS COMPLETE!")
print("="*70)
print("\n📁 All saved files are in the 'saved_models' directory:")
print("   - vgg19_model/")
print("   - inception_v3_model/")
print("   - vgg19_history.pkl")
print("   - inception_v3_history.pkl")
print("   - evaluation_metrics.json")
print("   - class_indices.json")